In [1]:
import h5py
import matplotlib.pyplot as plt
import yaml
import numpy as np
import argparse
import json
import tqdm
import io
import tarfile
import os

from matplotlib.patches import Rectangle
from matplotlib.collections import PatchCollection
from matplotlib import cm
from matplotlib.colors import Normalize
from statistics import mean, mode, stdev
from pathlib import Path

# data file with config read packets for dropped packets
#filename = "../data/packet-warm_ped_dropped_pkts.hdf5"

# only tile 10 powered up, warm test
#filename = "/data/v3/10x16/20260206_tile_4K04087_tests/data/warm/packet-warm_ped_480s_drops_t10_185vref.hdf5"
#filename = "/data/v3/10x16/20260206_tile_4K04087_tests/data/warm/packet-warm_ped_480s_drops_t10_220vref.hdf5"

# only tile 10 powered up, cold tests
filename = "/data/v3/10x16/20260206_tile_4K04087_tests/data/cold/packet-cold_ped_150kcycles_480s_t10_drop_pkts_1.hdf5"
#filename = "/data/v3/10x16/20260206_tile_4K04087_tests/data/cold/packet-cold_ped_150kcycles_480s_t10_drop_pkts_2.hdf5"
#filename = "/data/v3/10x16/20260206_tile_4K04087_tests/data/cold/packet-cold_ped_150kcycles_480s_t10_drop_pkts_3.hdf5"
#filename = "/data/v3/10x16/20260206_tile_4K04087_tests/data/cold/packet-cold_ped_150kcycles_480s_t10_drop_pkts_4.hdf5"
#filename = "/data/v3/10x16/20260206_tile_4K04087_tests/data/cold/packet-cold_ped_150kcycles_480s_t10_no_drop.hdf5"

# not-so-cold tests of tile 4K04087
#filename = "/data/v3/10x16/20260206_tile_4K04087_tests/cold/data/test_dropped_pkts_not_very_cold_1.hdf5"
#filename = "/data/v3/10x16/20260206_tile_4K04087_tests/cold/data/test_dropped_pkts_not_very_cold_2.hdf5"
#filename = "/data/v3/10x16/20260206_tile_4K04087_tests/cold/data/test_dropped_pkts_not_very_cold_3.hdf5"
#filename = "/data/v3/10x16/20260206_tile_4K04087_tests/cold/data/test_dropped_pkts_not_very_cold_4.hdf5"

# warm tests set 2 of 9 tiles
filename = "/data/v3/10x16/20260312_set2_10_tiles/data/warm/packet-warm_ped_540s_set2_9tiles_drop_pkts2.hdf5"

# warm tests of 10tiles
#filename = "/data/v3/10x16/20260214_set1_10_tiles/data/warm/packet-warm_ped_720s_no_hotchan_drops_220vref_3.hdf5"

# serial numbers for set 1 of tiles 1-10
chip_serial_number_list = ['', '4k02056', '4k03591', '4k01892', '4k02219', '4k02371', '4k04834', '4k00101', '4K00277', '4K03329', '4K04087' ]

# serial numbers for sets 2 and 3 
if "set2" in filename:
    chip_serial_number_list = ['', '4k04582', '4k04905', '4k03080', '4k04087', '4k04249', '4k00104', '4k01300', '4K04411', '4K01733', '4K00770' ]
elif "set3" in filename:
    chip_serial_number_list = ['', '4k00952', '4k03591', '4k03936', '4k00607', '4k00450', '4k04834', '4k00101', '4K00277', '4K03329', '4K03496' ]
elif "set4" in filename:
    chip_serial_number_list = ['', '4k01892', '4k02056', '4k02219', '4k02371', '4k00450', '4k04834', '4k00101', '4K00277', '4K03329', '4K03496' ]    


In [2]:
f = h5py.File(filename, 'r')
packets = f['packets'][:]

# create arrays of config read packets
read_mask     = f['packets'][:]['packet_type']  == 3
drop_pkts     = packets[read_mask]

iochs         = drop_pkts['io_channel'][:]
n_reads       = len(drop_pkts)

print(f"{n_reads} configuration packets were read")

for ioch in range(1,41):

    n_dropped = 0
    
    if ioch not in iochs:
        print(f"\nNo configuration read packet for IO_channel {ioch}")        

    else:
            
        ioch_mask = drop_pkts['io_channel'] == ioch
        ioch_pkts = drop_pkts[ioch_mask]
        
        ioch_chips = ioch_pkts['chip_id'][:]
        ioch_drops = ioch_pkts['register_data'][:]
        ioch_regis = ioch_pkts['register_address'][:]

        npkts = len(ioch_drops)
        print(f"\n{npkts} configuration read packets in IO_channel {ioch}")

        for pkt in range(npkts):
            if ioch_drops[pkt] > 0:
                print(f"\tIO_channel-Chip {ioch}-{ioch_chips[pkt]} logged {ioch_drops[pkt]} dropped pkts")
                n_dropped +=1

        print(f"\t{n_dropped} pkts dropped in IO_channel {ioch}")
    

1137 configuration packets were read

39 configuration read packets in IO_channel 1
	0 pkts dropped in IO_channel 1

39 configuration read packets in IO_channel 2
	0 pkts dropped in IO_channel 2

48 configuration read packets in IO_channel 3
	IO_channel-Chip 3-61 logged 1 dropped pkts
	1 pkts dropped in IO_channel 3

32 configuration read packets in IO_channel 4
	0 pkts dropped in IO_channel 4

39 configuration read packets in IO_channel 5
	0 pkts dropped in IO_channel 5

38 configuration read packets in IO_channel 6
	0 pkts dropped in IO_channel 6

48 configuration read packets in IO_channel 7
	0 pkts dropped in IO_channel 7

32 configuration read packets in IO_channel 8
	0 pkts dropped in IO_channel 8

38 configuration read packets in IO_channel 9
	0 pkts dropped in IO_channel 9

41 configuration read packets in IO_channel 10
	0 pkts dropped in IO_channel 10

73 configuration read packets in IO_channel 11
	IO_channel-Chip 11-63 logged 1 dropped pkts
	IO_channel-Chip 11-61 logged 1 dr